In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import hydra
import torch
from omegaconf import DictConfig

from genpp import BASE_DIR
from genpp.configs import register_resolvers

try:
    register_resolvers()
except Exception:
    print("Resolvers already registered:")

torch.set_float32_matmul_precision("high")

In [124]:
with hydra.initialize_config_dir(config_dir=str(BASE_DIR / "configs"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="base_engression",
        overrides=["trainer=debug", "model=cnn_engression_direct", "model.td_embedding_dim=8"],
    )

In [108]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()

Configuration hash: 1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c.pt.


In [126]:
model = hydra.utils.instantiate(
    cfg.model, rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None
)

In [127]:
datamodule.setup(stage="fit")
dataloader = datamodule.val_dataloader()
batch = next(iter(dataloader))

In [ ]:
res = model.forward(batch["x"], batch["timedelta"])
print(res.shape)

torch.Size([64, 8, 40, 32])


In [46]:
# model.compile(mode="max-autotune")
logger = False
trainer = hydra.utils.instantiate(cfg.trainer, logger=logger)
trainer.fit(model, datamodule)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You have turned on `Trainer(detect_anomaly=True)`. This will significantly slow down compute speed and is recommended only for model debugging.


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
`Trainer(overfit_batches=1)` was configured so 1 batch will be used.
/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/feik/GenPP/src/genpp/checkpoints exists and is not empty.











































Fitting fixed TD scaling: 100%|██████████| 1364/1364 [00:17<00:00, 80.11it/s] 

  | Name                | Type           | Params | Mode  | FLOPs
-----------------------------------------------------------------------
0 | backbone            | StochasticUNet | 408 K  | train | 0    
1 | crop                | CropND         | 0      | train | 0    
2 | loss_fn             | EnergyScore    | 0      | train | 0    
3 |

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:260: You requested to overfit but enabled train dataloader shuffling. We are turning off the train dataloader shuffling for you.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [47]:
# Try to load the model from the last checkpoint
ckpt_path = trainer.checkpoint_callback.best_model_path
# Get Model class
print(f"Loading model from checkpoint: {ckpt_path}")
model_loaded = model.__class__.load_from_checkpoint(ckpt_path, weights_only=False)

Loading model from checkpoint: /home/feik/GenPP/src/genpp/checkpoints/epoch=5-step=6.ckpt


In [48]:
trainer.predict(model_loaded, datamodule.val_dataloader(), return_predictions=False)

Predicting: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

In [49]:
datamodule.cleanup()